# Investigative EDA: feature leakage audit

Notebook-only analysis: imports pipeline code; does **not** modify production modules.

**Claim boundary:** Evidence for leakage-aware engineering (O2), core-model comparison (O3), and explanation sensitivity (O4). Not adversarial robustness or Feature Resilience Score.

**Privacy:** Outputs avoid raw `screen_name`, `description`, URLs, and profile text. Use integer row indices only for LIME examples.


In [1]:
# --- Setup: imports, repo root, output dir, seeds ---
from __future__ import annotations

import json
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from sklearn.feature_selection import mutual_info_classif


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for parent in [start, *start.parents]:
        if (parent / "DataLoader.py").exists():
            return parent
    raise FileNotFoundError(
        "Cannot find repository root (DataLoader.py). "
        "Execute from inside the repo or set cwd to the repo root."
    )


REPO_ROOT = find_repo_root()
repo_str = str(REPO_ROOT)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)

from DataLoader import load_twibot_splits_as_dict
from FeatureEngineering import BotFeatureExtractor, derive_reference_date
from Preprocessing import BotDetector
from benchmarking.data_prep import preprocess_split
from benchmarking.metrics import MetricsCalculator
from explainability.lime_explainer import LIMEExplainer
from explainability.shap_explainer import SHAPExplainer
from models.logistic_regression import LogisticRegressionModel
from models.random_forest import RandomForestModel
from models.xgboost import XGBoostModel

RANDOM_STATE = 2112
np.random.seed(RANDOM_STATE)


TS = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = REPO_ROOT / "results" / "eda_feature_leakage_audit" / TS
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {REPO_ROOT}")
print(f"Output dir: {OUTPUT_DIR}")

METRICS_CALC = MetricsCalculator()
MANIFEST: dict = {
    "timestamp_utc": TS,
    "repo_root": str(REPO_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "random_state": RANDOM_STATE,
}

def pkg_ver(name: str) -> str:
    try:
        import importlib.metadata as im
        return im.version(name)
    except Exception:
        return "unknown"

MANIFEST["packages"] = {
    "pandas": pkg_ver("pandas"),
    "numpy": pkg_ver("numpy"),
    "scikit-learn": pkg_ver("scikit-learn"),
    "xgboost": pkg_ver("xgboost"),
    "shap": pkg_ver("shap"),
    "lime": pkg_ver("lime"),
}


Repo root: C:\Users\ahmed\cs3ip-intelligent-bot-detection
Output dir: C:\Users\ahmed\cs3ip-intelligent-bot-detection\results\eda_feature_leakage_audit\20260430_134417


In [2]:
# --- Feature family mapping / helpers ---

def feature_family(name: str) -> str:
    if name in {"account_age_days", "is_verified"}:
        return "account"
    if name in {
        "followers_count", "friends_count", "listed_count", "statuses_count",
        "favourites_count", "followers_to_friends_ratio",
        "tweets_per_day", "favourites_per_day", "followers_per_day",
    }:
        return "activity"
    if name in {
        "default_profile", "default_profile_image", "has_extended_profile",
        "geo_enabled", "protected",
    }:
        return "profile_flags"
    if name in {
        "description_length", "has_description", "has_url",
        "screen_name_length", "screen_name_has_digits",
    }:
        return "text_proxy"
    return "other"


def ages_before_clip_stats(df: pd.DataFrame, reference_date) -> tuple[int, int]:
    """Count rows where raw age days would be <= 0 before extractor clip(lower=1)."""
    if reference_date is None or "account_creation_date" not in df.columns:
        return 0, len(df)
    acct = pd.to_datetime(df["account_creation_date"], errors="coerce")
    raw_days = (reference_date - acct).dt.days
    nonpos = int((raw_days <= 0).sum())
    return nonpos, len(df)


In [3]:
# --- Load official splits (train / dev→val / test) ---
DATA_DIR = REPO_ROOT / "data"
for fname in ("train.json", "dev.json", "test.json"):
    p = DATA_DIR / fname
    if not p.exists():
        raise FileNotFoundError(f"Missing required split file: {p}")

splits = load_twibot_splits_as_dict(DATA_DIR)

train_df = splits["train"].copy()
val_df = splits["val"].copy()
test_df = splits["test"].copy()

# Drop rows with missing labels (matches prepare_data guardrails)
for name, df in list(zip(["train", "val", "test"], [train_df, val_df, test_df])):
    if "label" not in df.columns:
        raise ValueError(f"No label column in {name}")
train_df = train_df.dropna(subset=["label"])
val_df = val_df.dropna(subset=["label"])
test_df = test_df.dropna(subset=["label"])

balance_rows = []
for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    vc = df["label"].astype(int).value_counts().sort_index()
    balance_rows.append({
        "split": split_name,
        "n_human_label_0": int(vc.get(0, 0)),
        "n_bot_label_1": int(vc.get(1, 0)),
        "n_total": len(df),
    })
balance_df = pd.DataFrame(balance_rows)
balance_path = OUTPUT_DIR / "class_balance_by_split.csv"
balance_df.to_csv(balance_path, index=False)
MANIFEST["class_balance_by_split"] = str(balance_path)
display(balance_df)

reference_date = derive_reference_date(train_df)
MANIFEST["reference_date_train_only"] = (
    reference_date.isoformat() if reference_date is not None else None
)
print(f"Reference date (train-only max account_creation_date): {reference_date}")

extractor = BotFeatureExtractor(reference_date=reference_date)
train_eng = extractor.extract_all_features(train_df.copy())
engineered_features = extractor.get_feature_names()
val_eng = extractor.extract_all_features(val_df.copy())
test_eng = extractor.extract_all_features(test_df.copy())

detector = BotDetector()
train_df_p = preprocess_split(detector, train_eng.copy())
val_df_p = preprocess_split(detector, val_eng.copy())
test_df_p = preprocess_split(detector, test_eng.copy())

model_numeric_features = (
    train_df_p.drop(columns=["label"]).select_dtypes(include=[np.number]).columns.tolist()
)
if not model_numeric_features:
    raise ValueError("No numeric features after preprocessing.")

extra_numeric = sorted(set(model_numeric_features) - set(engineered_features))
missing_cols_val = sorted(set(model_numeric_features) - set(val_df_p.columns))
missing_cols_test = sorted(set(model_numeric_features) - set(test_df_p.columns))
if missing_cols_val or missing_cols_test:
    raise ValueError(
        "Feature mismatch between splits after preprocessing. "
        f"Missing in val: {missing_cols_val}; missing in test: {missing_cols_test}"
    )

for split_name, raw_eng in [("train", train_eng), ("val", val_eng), ("test", test_eng)]:
    nonpos, n = ages_before_clip_stats(raw_eng, reference_date)
    rate = nonpos / n if n else 0.0
    print(f"account_age_days non-positive pre-clip rate [{split_name}]: {nonpos}/{n} ({rate:.4f})")

audit_21_features = list(engineered_features)
if len(audit_21_features) != 21:
    warnings.warn(
        f"Expected 21 engineered features for dissertation audit subset; got {len(audit_21_features)}. "
        "Continuing with extractor-defined list.",
        stacklevel=2,
    )

if extra_numeric:
    warnings.warn(
        f"model_numeric_features includes {len(extra_numeric)} columns not in BotFeatureExtractor list: "
        f"{extra_numeric[:10]}{'...' if len(extra_numeric) > 10 else ''}",
        stacklevel=2,
    )

HAS_VERIFIED = "is_verified" in audit_21_features
print(f"Engineered features: {len(engineered_features)} | Model numeric: {len(model_numeric_features)} | has is_verified: {HAS_VERIFIED}")


,split,n_human_label_0,n_bot_label_1,n_total
0,train,3632,4646,8278
1,val,1062,1303,2365
2,test,543,640,1183


Reference date (train-only max account_creation_date): 2020-09-04 07:55:38+00:00
account_age_days non-positive pre-clip rate [train]: 5/8278 (0.0006)
account_age_days non-positive pre-clip rate [val]: 1/2365 (0.0004)
account_age_days non-positive pre-clip rate [test]: 1/1183 (0.0008)
Engineered features: 21 | Model numeric: 24 | has is_verified: True


c:\Users\ahmed\cs3ip-intelligent-bot-detection\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: UserWarning: model_numeric_features includes 3 columns not in BotFeatureExtractor list: ['follower_sample_count', 'following_sample_count', 'tweet_count']
  exec(code_obj, self.user_global_ns, self.user_ns)


In [4]:
# --- Feature contract table ---
contract_rows = []
for f in engineered_features:
    contract_rows.append({
        "feature": f,
        "in_engineered_list": True,
        "in_model_numeric": f in model_numeric_features,
        "feature_family": feature_family(f),
    })
for f in extra_numeric:
    contract_rows.append({
        "feature": f,
        "in_engineered_list": False,
        "in_model_numeric": True,
        "feature_family": "loader_numeric_extra",
    })
contract_df = pd.DataFrame(contract_rows).sort_values(["in_engineered_list", "feature_family", "feature"])
contract_path = OUTPUT_DIR / "feature_contract_engineered_vs_model_numeric.csv"
contract_df.to_csv(contract_path, index=False)
MANIFEST["feature_contract"] = str(contract_path)
display(contract_df.head(30))


,feature,in_engineered_list,in_model_numeric,feature_family
21,follower_sample_count,False,True,loader_numeric_extra
22,following_sample_count,False,True,loader_numeric_extra
23,tweet_count,False,True,loader_numeric_extra
0,account_age_days,True,True,account
1,is_verified,True,True,account
6,favourites_count,True,True,activity
19,favourites_per_day,True,True,activity
2,followers_count,True,True,activity
20,followers_per_day,True,True,activity
7,followers_to_friends_ratio,True,True,activity


In [5]:
# --- Missingness / type / invalid audit (aggregate only; post-engineering pre-preprocess) ---
audit_sources = {
    "train": train_eng,
    "val": val_eng,
    "test": test_eng,
}

def invalid_counts(series: pd.Series) -> int:
    s = pd.to_numeric(series, errors="coerce")
    return int(s.isna().sum())

audit_rows = []
cols = [c for c in audit_21_features if c in train_eng.columns]
for feat in cols:
    row = {"feature": feat, "feature_family": feature_family(feat)}
    for split in ("train", "val", "test"):
        s = audit_sources[split][feat]
        n = len(s)
        miss = int(s.isna().sum())
        row[f"missing_pct_{split}"] = 100.0 * miss / n if n else 0.0
    # invalid / coercion: compare raw vs numeric coercion losses on train
    raw = audit_sources["train"][feat]
    inv = invalid_counts(raw)
    snum = pd.to_numeric(raw, errors="coerce")
    neg_ct = int((snum < 0).sum()) if snum.notna().any() else 0
    row["invalid_or_negative_count_train"] = inv + neg_ct
    row["type_coercion_required"] = str(raw.dtype) != "float64" and str(raw.dtype) != "int64"
    row["preprocessing_action"] = (
        "BotDetector.preprocess: numeric fillna(0); engineered extractor already coerced counts"
    )
    audit_rows.append(row)

missing_df = pd.DataFrame(audit_rows)
missing_path = OUTPUT_DIR / "missingness_type_invalid_audit.csv"
missing_df.to_csv(missing_path, index=False)
MANIFEST["missingness_audit"] = str(missing_path)
display(missing_df)


,feature,feature_family,missing_pct_train,missing_pct_val,missing_pct_test,invalid_or_negative_count_train,type_coercion_required,preprocessing_action
0,account_age_days,account,0.0,0.0,0.0,0,False,BotDetector.preprocess: numeric fillna(0); eng...
1,is_verified,account,0.0,0.0,0.0,0,False,BotDetector.preprocess: numeric fillna(0); eng...
2,followers_count,activity,0.0,0.0,0.0,0,False,BotDetector.preprocess: numeric fillna(0); eng...
3,friends_count,activity,0.0,0.0,0.0,0,False,BotDetector.preprocess: numeric fillna(0); eng...
4,listed_count,activity,0.0,0.0,0.0,0,False,BotDetector.preprocess: numeric fillna(0); eng...
5,statuses_count,activity,0.0,0.0,0.0,0,False,BotDetector.preprocess: numeric fillna(0); eng...
6,favourites_count,activity,0.0,0.0,0.0,0,False,BotDetector.preprocess: numeric fillna(0); eng...
7,followers_to_friends_ratio,activity,0.0,0.0,0.0,0,False,BotDetector.preprocess: numeric fillna(0); eng...
8,default_profile,profile_flags,0.0,0.0,0.0,0,False,BotDetector.preprocess: numeric fillna(0); eng...
9,default_profile_image,profile_flags,0.0,0.0,0.0,0,False,BotDetector.preprocess: numeric fillna(0); eng...


In [6]:
# --- Mutual information (train only, audit feature subset) ---
mi_feats = [c for c in audit_21_features if c in train_df_p.columns]
X_mi = train_df_p[mi_feats].replace([np.inf, -np.inf], np.nan).fillna(0)
y_mi = train_df_p["label"].astype(int).values

mi_scores = mutual_info_classif(X_mi, y_mi, random_state=RANDOM_STATE)
mi_df = pd.DataFrame({"feature": mi_feats, "mutual_information": mi_scores})
mi_df["feature_family"] = mi_df["feature"].map(feature_family)
mi_df = mi_df.sort_values("mutual_information", ascending=False)
mi_csv = OUTPUT_DIR / "mutual_information_train_only.csv"
mi_df.to_csv(mi_csv, index=False)
MANIFEST["mutual_information_csv"] = str(mi_csv)

top_n = min(15, len(mi_df))
plot_df = mi_df.head(top_n).iloc[::-1]
plt.figure(figsize=(10, 6))
sns.barplot(data=plot_df, x="mutual_information", y="feature", hue="feature_family", dodge=False)
plt.title(f"Top {top_n} features by mutual information (train only)")
plt.tight_layout()
mi_png = OUTPUT_DIR / "mutual_information_top_features.png"
plt.savefig(mi_png, dpi=150)
plt.close()
MANIFEST["mutual_information_png"] = str(mi_png)


In [7]:
# --- Verified sensitivity: LR / RF / XGBoost with & without is_verified ---

def prepare_xy(feature_list: list[str]):
    X_tr = train_df_p[feature_list].values.astype(np.float64)
    X_va = val_df_p[feature_list].values.astype(np.float64)
    X_te = test_df_p[feature_list].values.astype(np.float64)
    y_tr = train_df_p["label"].astype(int).values
    y_va = val_df_p["label"].astype(int).values
    y_te = test_df_p["label"].astype(int).values
    return X_tr, X_va, X_te, y_tr, y_va, y_te


def scale_lr(X_tr, X_va, X_te):
    det = BotDetector()
    return det.scale_features(X_tr, X_va, X_te)


models_cfg = [
    ("logistic_regression", lambda: LogisticRegressionModel(random_state=RANDOM_STATE)),
    ("random_forest", lambda: RandomForestModel(random_state=RANDOM_STATE)),
    ("xgboost", lambda: XGBoostModel(random_state=RANDOM_STATE)),
]

variants = [
    ("with_verified", audit_21_features),
]
if HAS_VERIFIED:
    wo = [c for c in audit_21_features if c != "is_verified"]
    variants.append(("without_verified", wo))
else:
    warnings.warn("is_verified not in model features; training single variant only.", stacklevel=2)

metric_records = []
for variant_name, feats in variants:
    X_tr, X_va, X_te, y_tr, y_va, y_te = prepare_xy(feats)
    for model_key, factory in models_cfg:
        mdl = factory()
        if model_key == "logistic_regression":
            X_tr_in, X_va_in, X_te_in = scale_lr(X_tr, X_va, X_te)
        else:
            X_tr_in, X_va_in, X_te_in = X_tr, X_va, X_te
        mdl.fit(X_tr_in, y_tr, feature_names=feats)
        for split_label, X_ev, y_ev in [
            ("val", X_va_in, y_va),
            ("test", X_te_in, y_te),
        ]:
            proba = mdl.predict_proba(X_ev)
            pred = mdl.predict(X_ev)
            met = METRICS_CALC.compute_all_metrics(y_ev, pred, y_proba=proba)
            row = {
                "variant": variant_name,
                "model": model_key,
                "eval_split": split_label,
                **met,
            }
            metric_records.append(row)

metrics_df = pd.DataFrame(metric_records)
metrics_path = OUTPUT_DIR / "verified_sensitivity_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)
MANIFEST["verified_sensitivity_metrics"] = str(metrics_path)

required_keys = ["f1_macro", "pr_auc", "recall", "false_positive_rate", "false_negatives", "false_positives"]
for k in required_keys:
    if k not in metrics_df.columns:
        raise KeyError(k)
if not np.isfinite(metrics_df[required_keys].astype(float)).all().all():
    raise ValueError("Non-finite metrics detected")

# Delta table: with_verified minus without_verified
delta_rows = []
if HAS_VERIFIED:
    base = metrics_df.set_index(["model", "eval_split", "variant"])
    for model_key in metrics_df["model"].unique():
        for split_label in ("val", "test"):
            try:
                w = base.loc[(model_key, split_label, "with_verified")]
                wo = base.loc[(model_key, split_label, "without_verified")]
            except KeyError:
                continue
            dr = {"model": model_key, "eval_split": split_label}
            for mk in required_keys + ["f1", "accuracy"]:
                if mk in w.index and mk in wo.index:
                    dr[f"delta_{mk}"] = float(w[mk]) - float(wo[mk])
            delta_rows.append(dr)
delta_df = pd.DataFrame(delta_rows)
delta_path = OUTPUT_DIR / "verified_sensitivity_deltas.csv"
delta_df.to_csv(delta_path, index=False)
MANIFEST["verified_sensitivity_deltas"] = str(delta_path)
display(metrics_df[["variant", "model", "eval_split", "f1_macro", "pr_auc", "recall", "false_positive_rate"]])


,variant,model,eval_split,f1_macro,pr_auc,recall,false_positive_rate
0,with_verified,logistic_regression,val,0.786913,0.824418,0.977744,0.411488
1,with_verified,logistic_regression,test,0.802267,0.823567,0.968750,0.370166
2,with_verified,random_forest,val,0.789577,0.833518,0.973139,0.402072
3,with_verified,random_forest,test,0.802760,0.853616,0.964063,0.364641
4,with_verified,xgboost,val,0.788210,0.834585,0.940138,0.371940
5,with_verified,xgboost,test,0.793119,0.846370,0.925000,0.344383
6,without_verified,logistic_regression,val,0.697194,0.752255,0.837299,0.444444
7,without_verified,logistic_regression,test,0.709354,0.765657,0.828125,0.410681
8,without_verified,random_forest,val,0.749712,0.799374,0.880276,0.386064
9,without_verified,random_forest,test,0.746168,0.830644,0.859375,0.370166


In [8]:
# --- Shortcut-risk triangulation: association + LR coefficient shift ---
shortcut_lines = []

if HAS_VERIFIED and "is_verified" in train_df_p.columns:
    iv = train_df_p["is_verified"].astype(int)
    y = train_df_p["label"].astype(int)
    tbl = pd.crosstab(iv, y)
    shortcut_lines.append("### Label association vs is_verified (train)\n")
    try:
        shortcut_lines.append(tbl.to_markdown())
    except ImportError:
        shortcut_lines.append(tbl.to_string())
    # Odds ratio (simple): P(bot|verified)/P(bot|unverified) approx from rates
    p_bot_v = y[iv == 1].mean() if (iv == 1).any() else np.nan
    p_bot_u = y[iv == 0].mean() if (iv == 0).any() else np.nan
    shortcut_lines.append(f"\nP(bot | verified=1): {p_bot_v:.4f}")
    shortcut_lines.append(f"P(bot | verified=0): {p_bot_u:.4f}")
else:
    shortcut_lines.append("is_verified absent; skip association block.")

# LR coefficients: refit stripped-down for coef extract
feats_w = audit_21_features
feats_wo = [c for c in feats_w if c != "is_verified"] if HAS_VERIFIED else feats_w

def lr_coef_dict(feature_list):
    X_tr, X_va, X_te, y_tr, y_va, y_te = prepare_xy(feature_list)
    X_tr_s, _, _ = scale_lr(X_tr, X_va, X_te)
    lr = LogisticRegressionModel(random_state=RANDOM_STATE)
    lr.fit(X_tr_s, y_tr, feature_names=feature_list)
    return lr.get_coefficients(), lr.get_odds_ratios()

coef_w, or_w = lr_coef_dict(feats_w)
coef_wo, or_wo = (None, None)
if HAS_VERIFIED:
    coef_wo, or_wo = lr_coef_dict(feats_wo)

coef_compare = []
for fname in sorted(set(coef_w) | set(coef_wo or {})):
    row = {"feature": fname, "coef_with_verified": coef_w.get(fname), "coef_without_verified": (coef_wo or {}).get(fname)}
    coef_compare.append(row)
coef_compare_df = pd.DataFrame(coef_compare)
shortcut_lines.append("\n### LR coefficient snapshot (is_verified row)\n")
_ivsnap = coef_compare_df[coef_compare_df["feature"] == "is_verified"]
try:
    shortcut_lines.append(_ivsnap.to_markdown(index=False) if len(_ivsnap) else "(no row)")
except ImportError:
    shortcut_lines.append(_ivsnap.to_string(index=False) if len(_ivsnap) else "(no row)")

shortcut_md = OUTPUT_DIR / "shortcut_risk_summary.md"
shortcut_md.write_text("\n".join(shortcut_lines), encoding="utf-8")
MANIFEST["shortcut_risk_summary"] = str(shortcut_md)


In [9]:
# --- SHAP global shift (RF + XGBoost); LR uses coefficients above ---
SHAP_MAX_SAMPLES = 100
rng = np.random.RandomState(RANDOM_STATE)

def shap_importance(model_cls, feats: list[str]) -> pd.Series:
    X_tr, X_va, X_te, y_tr, y_va, y_te = prepare_xy(feats)
    X_tr_in, X_va_in, X_te_in = X_tr, X_va, X_te
    m = model_cls()
    m.fit(X_tr_in, y_tr, feature_names=feats)
    bg = X_tr_in if len(X_tr_in) <= SHAP_MAX_SAMPLES else X_tr_in[rng.choice(len(X_tr_in), SHAP_MAX_SAMPLES, replace=False)]
    exp = SHAPExplainer(m, feature_names=feats).fit(bg, max_samples=SHAP_MAX_SAMPLES)
    X_explain = X_va_in if len(X_va_in) <= 500 else X_va_in[rng.choice(len(X_va_in), 500, replace=False)]
    exp.explain(X_explain)
    imp = exp.get_global_importance()
    return pd.Series(imp).sort_values(ascending=False)

shap_rows = []
for variant_name, feats in variants:
    for model_key, cls in [
        ("random_forest", RandomForestModel),
        ("xgboost", XGBoostModel),
    ]:
        try:
            s = shap_importance(cls, feats)
            for feat, val in s.items():
                shap_rows.append({"model": model_key, "variant": variant_name, "feature": feat, "mean_abs_shap": float(val)})
        except Exception as exc:
            warnings.warn(f"SHAP failed for {model_key}/{variant_name}: {exc}", stacklevel=2)

shap_df = pd.DataFrame(shap_rows)
if not shap_df.empty:
    pivot = shap_df.pivot_table(index=["model", "feature"], columns="variant", values="mean_abs_shap")
    pivot.columns.name = None
    rename_map = {
        "with_verified": "mean_abs_shap_with",
        "without_verified": "mean_abs_shap_without",
    }
    pivot = pivot.rename(columns={k: v for k, v in rename_map.items() if k in pivot.columns})
    shap_shift_path = OUTPUT_DIR / "shap_verified_shift_summary.csv"
    pivot.reset_index().to_csv(shap_shift_path, index=False)
    MANIFEST["shap_verified_shift_summary"] = str(shap_shift_path)
else:
    warnings.warn("No SHAP rows produced (optional deps or runtime).", stacklevel=2)


c:\Users\ahmed\cs3ip-intelligent-bot-detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# --- LIME: same comparison model on up to 3 false-negative bots (test) ---
# Pick RF vs XGB by test f1_macro (macro) with_verified variant
sub = metrics_df[(metrics_df["eval_split"] == "test") & (metrics_df["variant"] == "with_verified")]
if sub.empty:
    sub = metrics_df[metrics_df["eval_split"] == "test"]
best_row = sub.loc[sub["f1_macro"].idxmax()]
comparison_model_key = best_row["model"]
MANIFEST["lime_comparison_model"] = comparison_model_key

feats_w = audit_21_features
X_tr, X_va, X_te, y_tr, y_va, y_te = prepare_xy(feats_w)

if comparison_model_key == "logistic_regression":
    X_tr_in, X_va_in, X_te_in = scale_lr(X_tr, X_va, X_te)
    cmp_model = LogisticRegressionModel(random_state=RANDOM_STATE)
elif comparison_model_key == "random_forest":
    X_tr_in, X_va_in, X_te_in = X_tr, X_va, X_te
    cmp_model = RandomForestModel(random_state=RANDOM_STATE)
elif comparison_model_key == "xgboost":
    X_tr_in, X_va_in, X_te_in = X_tr, X_va, X_te
    cmp_model = XGBoostModel(random_state=RANDOM_STATE)
else:
    raise ValueError(f"Unexpected comparison model: {comparison_model_key}")

cmp_model.fit(X_tr_in, y_tr, feature_names=feats_w)
proba_te = cmp_model.predict_proba(X_te_in)[:, 1]
pred_te = cmp_model.predict(X_te_in)
y_te_arr = y_te.astype(int)

fn_mask = (y_te_arr == 1) & (pred_te == 0)
fn_positions = np.flatnonzero(fn_mask)

lime_summaries = []
fallback_reason = None

if len(fn_positions) == 0:
    fallback_reason = "no_false_negatives_used_lowest_confidence_true_bots"
    bot_mask = y_te_arr == 1
    bot_idx = np.flatnonzero(bot_mask)
    if len(bot_idx):
        order = np.argsort(proba_te[bot_idx])
        chosen = bot_idx[order[:3]]
    else:
        chosen = np.array([], dtype=int)
else:
    chosen = fn_positions[:3]

lime_expl = LIMEExplainer(cmp_model, feature_names=feats_w).fit(X_tr_in)

for pos in chosen:
    inst = X_te_in[pos]
    expl = lime_expl.explain_instance(inst, num_features=min(10, len(feats_w)), num_samples=3000)
    top_feats = sorted(expl["feature_contributions"].items(), key=lambda x: abs(x[1]), reverse=True)[:10]
    lime_summaries.append({
        "test_row_index": int(pos),
        "true_label": int(y_te_arr[pos]),
        "predicted_label": int(pred_te[pos]),
        "bot_proba": float(proba_te[pos]),
        "fallback_reason": fallback_reason or "",
        "top_lime_features_json": json.dumps({k: float(v) for k, v in top_feats}),
    })

lime_df = pd.DataFrame(lime_summaries)
lime_path = OUTPUT_DIR / "lime_selected_examples_summary.csv"
lime_df.to_csv(lime_path, index=False)
MANIFEST["lime_selected_examples_summary"] = str(lime_path)


In [11]:
# --- Spearman correlation clustered heatmap (train, audit features) ---
corr_feats = [c for c in audit_21_features if c in train_df_p.columns]
C = train_df_p[corr_feats].corr(method="spearman")

# Hierarchical clustering order
dist = 1 - C.fillna(0).abs()
np.fill_diagonal(dist.values, 0)
condensed = squareform(dist.values, checks=False)
linkage = hierarchy.linkage(condensed, method="average")
order = hierarchy.leaves_list(linkage)
cols_ordered = [corr_feats[i] for i in order]
C_ord = C.loc[cols_ordered, cols_ordered]

corr_csv = OUTPUT_DIR / "spearman_correlation_matrix.csv"
C_ord.to_csv(corr_csv)
MANIFEST["spearman_correlation_matrix"] = str(corr_csv)

g = sns.clustermap(C.fillna(0), cmap="vlag", vmin=-1, vmax=1, figsize=(12, 12))
g.savefig(OUTPUT_DIR / "spearman_clustered_heatmap.png", dpi=150, bbox_inches="tight")
plt.close("all")

MANIFEST["spearman_clustered_heatmap"] = str(OUTPUT_DIR / "spearman_clustered_heatmap.png")


In [12]:
# --- Manifest + dissertation-ready bullets ---
manifest_path = OUTPUT_DIR / "notebook_execution_manifest.json"
artifact_names = [
    "class_balance_by_split.csv",
    "feature_contract_engineered_vs_model_numeric.csv",
    "missingness_type_invalid_audit.csv",
    "mutual_information_train_only.csv",
    "mutual_information_top_features.png",
    "verified_sensitivity_metrics.csv",
    "verified_sensitivity_deltas.csv",
    "shortcut_risk_summary.md",
    "shap_verified_shift_summary.csv",
    "lime_selected_examples_summary.csv",
    "spearman_correlation_matrix.csv",
    "spearman_clustered_heatmap.png",
]
missing_art = [n for n in artifact_names if not (OUTPUT_DIR / n).exists()]
if missing_art:
    warnings.warn(f"Missing artifacts before manifest write: {missing_art}", stacklevel=2)
MANIFEST["artifacts"] = [str(OUTPUT_DIR / n) for n in artifact_names]
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(MANIFEST, f, indent=2)

print("DONE. Summary bullets:")
print("- Official splits loaded; dev.json mapped to val; reference_date from train only.")
print("- Feature contract distinguishes engineered vs model numeric columns.")
print("- Metrics finite check passed for sensitivity table.")
print("- Outside scope: adversarial perturbations, flip-rate robustness, Feature Resilience Score (see benchmarking/robustness.py).")
print(f"Manifest: {manifest_path}")


DONE. Summary bullets:
- Official splits loaded; dev.json mapped to val; reference_date from train only.
- Feature contract distinguishes engineered vs model numeric columns.
- Metrics finite check passed for sensitivity table.
- Outside scope: adversarial perturbations, flip-rate robustness, Feature Resilience Score (see benchmarking/robustness.py).
Manifest: C:\Users\ahmed\cs3ip-intelligent-bot-detection\results\eda_feature_leakage_audit\20260430_134417\notebook_execution_manifest.json
